In [ ]:
!pip install -q "pyarrow<24,>=19.0.0" transformers datasets accelerate peft bitsandbytes scikit-learn
import os, re, time
from collections import Counter
import numpy as np, pandas as pd, torch
from google.colab import drive
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/valid.csv"
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
SEED, MAX_LEN = 42, 448  # Room for every lead+local prompt + 32-token target (320 cut off ~12% of prompts)
torch.manual_seed(SEED); np.random.seed(SEED)


Mounted at /content/drive


In [ ]:
# 1. Filtered Multi-Sense Ambiguous Split (In-Memory, Zero Drive Writes)
df = pd.read_csv(CSV_PATH, nrows=300000).dropna(subset=["TEXT", "LOCATION", "LABEL"])
df["LOCATION"] = df["LOCATION"].astype(int)
df["ACRONYM"] = [w[l] if 0 <= l < len(w) else "" for w, l in zip(df["TEXT"].str.split(), df["LOCATION"])]
df = df[df["ACRONYM"] != ""]

ambig = df[df["ACRONYM"].isin(df.groupby("ACRONYM")["LABEL"].nunique()[lambda x: x >= 2].index)]
ambig = ambig[ambig["LABEL"].isin(ambig["LABEL"].value_counts()[lambda x: x >= 30].index)]
top40 = ambig.groupby("ACRONYM").filter(lambda g: g["LABEL"].nunique() >= 2)["ACRONYM"].value_counts().head(40).index.tolist()

np.random.shuffle(top40)
seen_pool = ambig[ambig["ACRONYM"].isin(top40[:30])]
unseen_pool = ambig[ambig["ACRONYM"].isin(top40[30:])]

sampled_seen, _ = train_test_split(seen_pool, train_size=1500, stratify=seen_pool["LABEL"], random_state=SEED)
train_df, test_seen_df = train_test_split(sampled_seen, train_size=1000, test_size=500, stratify=sampled_seen["LABEL"], random_state=SEED)
test_unseen_df = unseen_pool.sample(n=min(500, len(unseen_pool)), random_state=SEED)


In [ ]:
# 2. Lead + Local Context Window & Prompt Formatting
SYS = "You are a clinical and biomedical text parser. Expand the bracketed abbreviation based on the context. Output ONLY the exact canonical expansion string in lowercase, with no punctuation, quotes, or explanation."
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.pad_token or tok.eos_token

def get_prompt(row, lead=40, win=60):
    w, loc = str(row["TEXT"]).split(), int(row["LOCATION"])
    abbr = w[loc] if 0 <= loc < len(w) else row["ACRONYM"]
    if 0 <= loc < len(w): w[loc] = f"[{abbr}]"
    # Never truncate the tokenized prompt (that cuts off "Expansion:" and the assistant header);
    # if it is too long, narrow the local window instead.
    while True:
        s, e = max(0, loc - win), min(len(w), loc + win + 1)
        ctx = f"{' '.join(w[:lead])} ... {' '.join(w[s:e])}" if s > lead else " ".join(w[:e])
        msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": f"Abbreviation: [{abbr}]\nContext: {ctx}\nExpansion:"}]
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        if len(tok(prompt, add_special_tokens=False)["input_ids"]) <= MAX_LEN - 32 or win <= 5:
            return prompt
        win -= 5

def encode_row(row):
    p_ids = tok(get_prompt(row), add_special_tokens=False)["input_ids"]
    t_ids = tok(f"{str(row['LABEL']).strip().lower()}{tok.eos_token}", truncation=True, max_length=32, add_special_tokens=False)["input_ids"]
    ids = p_ids + t_ids
    return {"input_ids": ids, "labels": [-100] * len(p_ids) + t_ids, "attention_mask": [1] * len(ids)}

train_ds = Dataset.from_list([encode_row(r) for _, r in train_df.iterrows()])


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

In [ ]:
# 3. 4-Bit QLoRA + NEFTune Training
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = prepare_model_for_kbit_training(AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto"))
lora = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], lora_dropout=0.10, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)

args = TrainingArguments(
    output_dir="/content/tmp_lora", per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=1.5e-4, num_train_epochs=3, lr_scheduler_type="cosine", warmup_steps=0.1,
    weight_decay=0.01, neftune_noise_alpha=5.0, fp16=True, logging_steps=25, save_strategy="no", report_to="none", seed=SEED
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8))

torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train(); train_time = time.time() - t0


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,2.599405
50,1.268087
75,0.536039
100,0.389434
125,0.330392
150,0.121177
175,0.099196


In [ ]:
# 4. 3-Beam Search Evaluation (EM, Macro F1, Token F1)
model.eval(); tok.padding_side = "left"
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^\w\s-]", "", str(s).lower())).strip()

def tok_f1(p, t):
    pt, tt = p.split(), t.split()
    if not pt or not tt: return float(pt == tt)
    c = sum((Counter(pt) & Counter(tt)).values())
    return (2 * c / (len(pt) + len(tt))) if c else 0.0

def evaluate(df_split, bs=16):
    prompts = [get_prompt(r) for _, r in df_split.iterrows()]
    preds, t_start = [], time.time()
    with torch.no_grad():
        for i in range(0, len(prompts), bs):
            inp = tok(prompts[i:i+bs], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)  # template already has <|begin_of_text|>
            out = model.generate(**inp, max_new_tokens=20, num_beams=3, early_stopping=True, do_sample=False, pad_token_id=tok.eos_token_id)
            preds.extend(norm(tok.decode(seq[inp["input_ids"].shape[1]:], skip_special_tokens=True)) for seq in out)
    targs = [norm(l) for l in df_split["LABEL"]]
    return accuracy_score(targs, preds), f1_score(targs, preds, average="macro", zero_division=0), np.mean([tok_f1(p, t) for p, t in zip(preds, targs)]), time.time() - t_start

s_em, s_mf1, s_tf1, s_t = evaluate(test_seen_df)
u_em, u_mf1, u_tf1, u_t = evaluate(test_unseen_df)
vram = torch.cuda.max_memory_allocated() / 1024**2

print(f"\nTrain: {train_time:.1f}s | Latency: {s_t/len(test_seen_df)*1000:.1f}ms | VRAM: {vram:.0f}MB")
print(f"Seen   -> EM: {s_em*100:.2f}% | Macro F1: {s_mf1*100:.2f}% | Token F1: {s_tf1*100:.2f}%")
print(f"Unseen -> EM: {u_em*100:.2f}% | Macro F1: {u_mf1*100:.2f}% | Token F1: {u_tf1*100:.2f}%")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and 


Train: 706.0s | Latency: 284.1ms | VRAM: 5896MB
Seen   -> EM: 70.80% | Macro F1: 53.53% | Token F1: 73.60%
Unseen -> EM: 12.00% | Macro F1: 3.36% | Token F1: 18.10%
